# Modesty Blur: train the person classifier (Kaggle)

**Before you run:** (1) Notebook settings (right side) > **Accelerator: GPU T4 x2**, **Internet: On**. (2) **Add Input** > Your Datasets > pick the dataset you uploaded (the six zip files).

Then press **Run All**. When it finishes, the two result files are in the **Output** panel (right side, `/kaggle/working/out`): download `person_cls.onnx` and `person_cls.json` and attach them to the chat with Claude.

Your pictures stay in your private Kaggle dataset.

In [ ]:
#@title 1. Get the code
!rm -rf /kaggle/working/repo
!git clone -q --depth 1 https://github.com/Nex-Gen-Dev/Modesty-blur /kaggle/working/repo
!pip -q install onnx onnxruntime
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - turn on the GPU in the notebook settings')

In [ ]:
import os, zipfile, shutil
CLASSES = ['woman', 'man', 'girl', 'boy', 'teen_girl', 'teen_boy']
IN, OUT = '/kaggle/input', '/kaggle/working/data'
IMG = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

def find_sources(c):
    dirs, zips = [], []
    for root, ds, fs in os.walk(IN):
        for d in ds:
            if d.lower() == c:
                dirs.append(os.path.join(root, d))
        for f in fs:
            if f.lower() == c + '.zip':
                zips.append(os.path.join(root, f))
    dirs = [d for d in dirs if not any(d != o and d.startswith(o + os.sep) for o in dirs)]   # skip nested copies
    return dirs, zips

def collect(c, out_root):
    dst = os.path.join(out_root, c)
    os.makedirs(dst, exist_ok=True)
    dirs, zips = find_sources(c)
    k = 0
    for d in dirs:
        for root, _, fs in os.walk(d):
            for f in fs:
                if f.lower().endswith(IMG):
                    shutil.copy(os.path.join(root, f), os.path.join(dst, f'{k:07d}' + os.path.splitext(f)[1].lower()))
                    k += 1
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            for name in zf.namelist():
                if name.lower().endswith(IMG) and '__MACOSX' not in name:
                    with zf.open(name) as s, open(os.path.join(dst, f'{k:07d}' + os.path.splitext(name)[1].lower()), 'wb') as o:
                        shutil.copyfileobj(s, o)
                    k += 1
    return k

for c in CLASSES:
    n = collect(c, OUT)
    print(f'{c:10s}: {n} pictures', '   <-- NONE FOUND' if n == 0 else '')
if any(not os.listdir(os.path.join(OUT, c)) for c in CLASSES):
    print('\nWhat the notebook can see in /kaggle/input (check the names):')
    for root, ds, fs in os.walk(IN):
        if root.count(os.sep) <= 5:
            print(root, ds[:8], fs[:5])

In [ ]:
#@title 3. Train (a few minutes for ~3,000 pictures)
EPOCHS = 15
FEMALE_WEIGHT = 3.0   # how much worse it is to miss a woman/girl than to blur a man/boy
!cd /kaggle/working/repo && python tools/train_person_cls.py /kaggle/working/data --out /kaggle/working/out --epochs {EPOCHS} --female-weight {FEMALE_WEIGHT} --workers 4 --batch 64 --shrink /kaggle/working/small

In [ ]:
import os
print(os.listdir('/kaggle/working/out'))
print('Download person_cls.onnx and person_cls.json from the Output panel (right side, /kaggle/working/out), then attach them in the chat with Claude.')